In [ ]:
"""
Based on the paper of Simpson & Woolway (2021)
https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2020WR029441
"""

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt

import sys
sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config
from utils_energy_analysis import *

In [ ]:
base_folder = r"/storage/alplakes_test/lucerne_100m_2025"

In [ ]:
bin_folder = os.path.join(r'/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/lucerne_100m_2025', 'binary_data')

# Import datasets

In [ ]:
ds_wind = xr.open_dataset(os.path.join(bin_folder, "wind.nc"))

In [ ]:
model = 'lucerne_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

In [ ]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC']) + 1) * grid_resolution - grid_resolution / 2
ds['XC'] = np.arange(1, len(ds['XC']) + 1) * grid_resolution - grid_resolution / 2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

In [ ]:
mask = ds['THETA'].isel(time=0).values > 0

# Wind stress components ($\tau_x$ & $\tau_y$)

### Cd

In [ ]:
# Following the MITgcm calculation
drag_coeff1 = 0.00135
drag_coeff2 = 0.000071
drag_coeff3 = 0.0000382

def compute_cd(wind_speed):
    cd = drag_coeff1 / wind_speed + drag_coeff2 + drag_coeff3 * wind_speed
    return cd

In [ ]:
test_speed_arr = np.array([0.5, 1, 2 ,3, 4, 5, 6, 7, 10, 12])
plt.plot(test_speed_arr, compute_cd(test_speed_arr))

In [ ]:
cd = compute_cd(ds_wind.speed)

# $\tau_x$ & $\tau_y$

In [ ]:
rho_a = 1,225 # 15°C

In [ ]:
tau_x = ds_wind.u10 * cd * rho_a

In [ ]:
tau_y = ds_wind.v10 * cd * rho_a

# Wind Rate of Working

In [ ]:
ds_surface = ds.isel(Z=0)

In [ ]:
RW = tau_x * ds_surface.UVEL + tau_y * ds_surface.VVEL

In [ ]:
#